# 191. τ-bench：怎样评测有状态的 Tool-Agent-User 回合？

> **面试问题：如何同时验证最终数据库状态、用户确认、领域策略和多次运行可靠性，而不是只看 Agent 最后一段话？**

## 先给结论

高质量回答需要同时说明目标状态、可执行策略、状态版本、失败回滚和可复放评测。下面不用 Agent 框架或远程工具，而是用受控的内存模型把核心合同写出来；断言只证明教学实现的不变量，不能替代真实服务的隔离、审计、权限与压测。

## 一手资料

- [τ-bench](https://arxiv.org/abs/2406.12045)
- [AgentBench](https://arxiv.org/abs/2308.03688)
- [Toolformer](https://arxiv.org/abs/2302.04761)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "needs-isolation"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "isolation" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：为什么只看最终自然语言回答不够

工具 Agent 的回答会改变订单、账户或工单状态。τ-bench 的关键思想是让用户、Agent、领域 API 和策略规则一起构成一个有状态回合，并在回合结束时检查目标数据库状态。这里用订单退款模拟该合同：订单归属、订单状态和用户确认缺一不可。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass  # 执行本行的状态、计算或校验逻辑。
class Order:  # 执行本行的状态、计算或校验逻辑。
    order_id: str  # 执行本行的状态、计算或校验逻辑。
    owner: str  # 执行本行的状态、计算或校验逻辑。
    status: str  # 执行本行的状态、计算或校验逻辑。
    refunded: bool = False  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class ToolCall:  # 执行本行的状态、计算或校验逻辑。
    name: str  # 执行本行的状态、计算或校验逻辑。
    order_id: str  # 执行本行的状态、计算或校验逻辑。
orders = {"o-1": Order("o-1", "alice", "delivered")}  # 执行本行的状态、计算或校验逻辑。
assert orders["o-1"].refunded is False  # 执行本行的状态、计算或校验逻辑。
assert orders["o-1"].owner == "alice"  # 执行本行的状态、计算或校验逻辑。
assert orders["o-1"].status == "delivered"  # 执行本行的状态、计算或校验逻辑。


## 2. 策略门禁：先检查权限和前置状态

策略不是提示词中的一段建议，而应是动作执行前的确定性 gate。真实产品会把金额阈值、账户权限、地区法规和人工确认放在策略引擎；演示只保留“本人、已送达、已确认”三个可审计条件。


In [ ]:
def allowed_refund(order, user, confirmed):  # 执行本行的状态、计算或校验逻辑。
    return order.owner == user and order.status == "delivered" and confirmed  # 执行本行的状态、计算或校验逻辑。
assert allowed_refund(orders["o-1"], "alice", True)  # 执行本行的状态、计算或校验逻辑。
assert not allowed_refund(orders["o-1"], "bob", True)  # 执行本行的状态、计算或校验逻辑。
assert not allowed_refund(orders["o-1"], "alice", False)  # 执行本行的状态、计算或校验逻辑。


## 3. 领域工具：工具只接收结构化调用

Agent 应先将文本决策编译为可验证的结构化参数，再交给工具。工具自身也必须防御：即使上游策略漏检，工具仍拒绝不存在的订单、错误状态或重复退款。日志是评测与事后调查的一部分。


In [ ]:
class RetailEnv:  # 执行本行的状态、计算或校验逻辑。
    def __init__(self, order_map):  # 执行本行的状态、计算或校验逻辑。
        self.orders = order_map  # 执行本行的状态、计算或校验逻辑。
        self.audit = []  # 执行本行的状态、计算或校验逻辑。
    def execute(self, call):  # 执行本行的状态、计算或校验逻辑。
        order = self.orders.get(call.order_id)  # 执行本行的状态、计算或校验逻辑。
        if call.name != "refund" or order is None or order.refunded:  # 执行本行的状态、计算或校验逻辑。
            raise ValueError("工具调用不满足领域不变量")  # 执行本行的状态、计算或校验逻辑。
        order.refunded = True  # 执行本行的状态、计算或校验逻辑。
        self.audit.append((call.name, call.order_id))  # 执行本行的状态、计算或校验逻辑。
        return {"ok": True, "state": "refunded"}  # 执行本行的状态、计算或校验逻辑。
env = RetailEnv(orders)  # 执行本行的状态、计算或校验逻辑。
assert env.audit == []  # 执行本行的状态、计算或校验逻辑。
assert env.orders["o-1"].status == "delivered"  # 执行本行的状态、计算或校验逻辑。


## 4. Agent loop：计划、门禁、调用、观察

loop 的每一步都应留下可复放事件：解析出的意图、候选 tool call、策略结论、工具观察。这里特意让拒绝分支不触碰订单状态，避免“先执行、再解释”为安全策略。


In [ ]:
def run_episode(env, user, order_id, confirmed):  # 执行本行的状态、计算或校验逻辑。
    order = env.orders.get(order_id)  # 执行本行的状态、计算或校验逻辑。
    call = ToolCall("refund", order_id)  # 执行本行的状态、计算或校验逻辑。
    if order is None or not allowed_refund(order, user, confirmed):  # 执行本行的状态、计算或校验逻辑。
        return {"ok": False, "reason": "policy_denied", "call": call}  # 执行本行的状态、计算或校验逻辑。
    observation = env.execute(call)  # 执行本行的状态、计算或校验逻辑。
    return {"ok": observation["ok"], "reason": "completed", "call": call}  # 执行本行的状态、计算或校验逻辑。
result = run_episode(env, "alice", "o-1", True)  # 执行本行的状态、计算或校验逻辑。
assert result["ok"] is True  # 执行本行的状态、计算或校验逻辑。
assert orders["o-1"].refunded is True  # 执行本行的状态、计算或校验逻辑。
assert env.audit == [("refund", "o-1")]  # 执行本行的状态、计算或校验逻辑。


## 5. 失败 trace：正确文本不能抵消错误动作

评测至少分离任务成功与策略遵从。下面构造未确认退款：Agent 可以提出合理的退款想法，但系统应该拒绝并且不产生新 audit 记录。这个反例能防止只看自然语言 explanation 的假阳性。


In [ ]:
before = len(env.audit)  # 执行本行的状态、计算或校验逻辑。
denied = run_episode(env, "alice", "o-1", False)  # 执行本行的状态、计算或校验逻辑。
assert denied["ok"] is False  # 执行本行的状态、计算或校验逻辑。
assert denied["reason"] == "policy_denied"  # 执行本行的状态、计算或校验逻辑。
assert len(env.audit) == before  # 执行本行的状态、计算或校验逻辑。


## 6. 最终状态 oracle：比较状态而非文案

真正的任务评测应把结束状态同预先标注的 goal state 比较。对于复杂 API，这通常包含多张表、字段和值域；演示只比较退款布尔值和仅一次工具调用，仍能说明“文本正确”和“状态正确”是不同信号。


In [ ]:
def goal_reached(env, order_id, expected_refunded, expected_calls):  # 执行本行的状态、计算或校验逻辑。
    order_ok = env.orders[order_id].refunded == expected_refunded  # 执行本行的状态、计算或校验逻辑。
    calls_ok = len(env.audit) == expected_calls  # 执行本行的状态、计算或校验逻辑。
    return order_ok and calls_ok  # 执行本行的状态、计算或校验逻辑。
assert goal_reached(env, "o-1", True, 1)  # 执行本行的状态、计算或校验逻辑。
assert not goal_reached(env, "o-1", False, 1)  # 执行本行的状态、计算或校验逻辑。
assert not goal_reached(env, "o-1", True, 2)  # 执行本行的状态、计算或校验逻辑。


## 7. 可靠性：一次成功不等于稳定成功

τ-bench 提出 pass^k 来观察多次尝试是否都成功。这里把每一次回合的状态/策略双重结果压成布尔值；生产环境还要按场景、工具、金额区间和随机种子切片，而不是只报一个平均数。


In [ ]:
def pass_at_k(trials, k):  # 执行本行的状态、计算或校验逻辑。
    if k <= 0 or len(trials) < k:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("k 必须落在可用试次范围")  # 执行本行的状态、计算或校验逻辑。
    return all(trials[:k])  # 执行本行的状态、计算或校验逻辑。
trials = [True, True, False]  # 执行本行的状态、计算或校验逻辑。
assert pass_at_k(trials, 1) is True  # 执行本行的状态、计算或校验逻辑。
assert pass_at_k(trials, 2) is True  # 执行本行的状态、计算或校验逻辑。
assert pass_at_k(trials, 3) is False  # 执行本行的状态、计算或校验逻辑。


## 8. 制品合同：让回合可以被复放与归因

应版本化策略、工具 schema、初始状态和评测目标；否则同一 trace 在新策略或新 API 上无法解释。这里只生成一个不可变摘要，生产中还应保存受控访问的输入、响应和权限快照。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"policy": "refund-v1", "tool_schema": "retail-v1", "goal": "o-1:refunded", "trial_count": 3}  # 执行本行的状态、计算或校验逻辑。
digest = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert len(digest) == 64  # 执行本行的状态、计算或校验逻辑。
assert artifact["tool_schema"] == "retail-v1"  # 执行本行的状态、计算或校验逻辑。
assert artifact["trial_count"] == len(trials)  # 执行本行的状态、计算或校验逻辑。


## 面试收束

按“任务目标 → 显式状态 → 动作前策略门禁 → 成功 oracle → 失败和重试 → 指标与版本化制品”的顺序回答。不要把一次文本看起来合理的演示当成可靠性证明：要独立检查状态、权限、不可逆副作用与多次运行的一致性。
